<a href="https://colab.research.google.com/github/mahendar8578/IntianalCommit/blob/main/Internal_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Based Stock Market Analysis and Prediction System

**Course/Assignment:** AI FOR FINANCE CIA-2  
**Project Choice:** Option 2: AI-Based Stock Market Analysis and Prediction System  

### Project Overview
This project leverages Artificial Intelligence and Machine Learning to analyze historical stock market data, identify critical trends, calculate technical indicators, and predict future price movements. It features an interactive interface where users can choose a stock ticker and analyze predictions dynamically.

In [ ]:
# Step 1: Install and Import Necessary Libraries
!pip install yfinance -q

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import ipywidgets as widgets
from ipywidgets import interact
import warnings
warnings.filterwarnings('ignore')

### Step 2: Data Fetching & Feature Engineering Engine
We will define a robust function that pulls live stock data using `yfinance` and engineers powerful technical indicators such as Moving Averages, Relative Strength Index (RSI), and Daily Returns to feed into our Machine Learning model.

In [ ]:
def get_stock_data(ticker, start_date='2020-01-01', end_date='2024-01-01'):
    # Fetch data
    df = yf.download(ticker, start=start_date, end=end_date)
    if df.empty:
        raise ValueError("No data found for this ticker symbol.")

    # Flatten MultiIndex columns if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Basic Features
    df['MA10'] = df['Close'].rolling(window=10).mean()
    df['MA50'] = df['Close'].rolling(window=50).mean()
    df['Daily_Return'] = df['Close'].pct_change()

    # RSI Indicator
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df['RSI'] = 100 - (100 / (1 + rs))

    # Target Variable: Next Day's Close Price
    df['Target_Close'] = df['Close'].shift(-1)

    # Drop NaNs
    df.dropna(inplace=True)
    return df

### Step 3: AI Model Training Engine
We train a **Random Forest Regressor** to learn patterns from historical close prices, volumes, moving averages, and technical indicators to forecast the next day's stock price.

In [ ]:
def train_ai_model(df):
    features = ['Close', 'Open', 'High', 'Low', 'Volume', 'MA10', 'MA50', 'RSI', 'Daily_Return']
    X = df[features]
    y = df['Target_Close']

    # Split data chronologically for financial time series
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Initialize and fit the Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate the model
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return model, X_test, y_test, predictions, rmse, r2

### Step 4: Interactive Application Dashboard
Run the cell below to load the interactive widget application. You can select preloaded popular financial stocks, visualize trends, and see the model's performance instantly.

In [ ]:
def run_dashboard(ticker):
    print(f"Fetching and processing data for {ticker}...")
    try:
        df = get_stock_data(ticker)
        model, X_test, y_test, predictions, rmse, r2 = train_ai_model(df)

        # Create Plots
        fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

        # Plot 1: Historical Prices and Moving Averages
        axes[0].plot(df.index, df['Close'], label='Close Price', color='blue', alpha=0.6)
        axes[0].plot(df.index, df['MA10'], label='10-Day MA (Short term)', color='orange', linestyle='--')
        axes[0].plot(df.index, df['MA50'], label='50-Day MA (Long term)', color='green', linestyle='-.')
        axes[0].set_title(f"{ticker} Stock Price & Moving Averages", fontsize=14)
        axes[0].set_ylabel("Price (USD)")
        axes[0].legend()
        axes[0].grid(True)

        # Plot 2: AI Predictions vs Real Values on Test Set
        test_dates = df.index[-len(y_test):]
        axes[1].plot(test_dates, y_test.values, label='Actual Next-Day Price', color='black', alpha=0.8)
        axes[1].plot(test_dates, predictions, label='AI Predicted Next-Day Price', color='red', linestyle='--')
        axes[1].set_title(f"AI Model Validation & Performance on Test Set", fontsize=14)
        axes[1].set_xlabel("Date")
        axes[1].set_ylabel("Price (USD)")
        axes[1].legend()
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()

        # Metrics Output Box
        print("\n" + "="*50)
        print(f" AI PERFORMANCE METRICS FOR {ticker}")
        print("="*50)
        print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
        print(f"Model R-squared (R²) Accuracy: {r2*100:.2f}%")

        # Generate live trading signal based on latest prediction
        latest_features = X_test.iloc[[-1]]
        latest_close = latest_features['Close'].values[0]
        predicted_next_close = model.predict(latest_features)[0]
        pct_change = ((predicted_next_close - latest_close) / latest_close) * 100

        print("\n" + "="*50)
        print(" LIVE INVESTMENT / TRADING RECOMMENDATION ")
        print("="*50)
        print(f"Latest Close Price: ${latest_close:.2f}")
        print(f"Predicted Next-Day Close: ${predicted_next_close:.2f} ({pct_change:+.2f}%)")
        if pct_change > 1.0:
            print("Signal: STRONG BUY (AI expects prices to rise significantly)")
        elif pct_change > 0:
            print("Signal: HOLD/NEUTRAL-BUY (Moderate positive momentum expected)")
        else:
            print("Signal: SELL/AVOID (AI expects potential correction/downward trend)")
        print("="*50)

    except Exception as e:
        print(f"Error processing ticker {ticker}: {e}")

# Widget interaction
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NFLX']
interact(run_dashboard, ticker=widgets.Dropdown(options=tickers, value='AAPL', description='Stock:'));

interactive(children=(Dropdown(description='Stock:', options=('AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NFLX')…